In [1]:
# If OpenCV isn't in your env run once:
# !pip install opencv-python-headless

import numpy as np
import cv2 as cv
from pathlib import Path
from IPython.display import HTML, Video
from pde_utils import correlated_gaussian_field, laplacian


In [2]:
def find_wave_fronts(gray: np.ndarray, k: int = 10) -> np.ndarray:
    """Return k (x,y) points with strongest gradient magnitude."""
    ddepth = cv.CV_16S
    gx = cv.Sobel(gray, ddepth, 1, 0, ksize=3)
    gy = cv.Sobel(gray, ddepth, 0, 1, ksize=3)
    grad = cv.addWeighted(cv.convertScaleAbs(gx), 0.5,
                          cv.convertScaleAbs(gy), 0.5, 0)
    best = np.argsort(grad.ravel())[-k:]
    pts  = np.asarray([np.unravel_index(i, gray.shape)[::-1] for i in best],
                      np.float32)              # (x,y)
    return pts[:, None, :]                     # (k,1,2) for LK

def to_uint8(stack: np.ndarray) -> np.ndarray:
    vmin, vmax = stack.min(), stack.max()
    return (255*(stack - vmin)/(vmax - vmin + 1e-12)).astype(np.uint8)

def save_mp4(frames, path: str, fps=10):
    h, w = frames[0].shape[:2]
    vw = cv.VideoWriter(path,
                        cv.VideoWriter_fourcc(*'mp4v'),
                        fps, (w, h), True)
    for f in frames:
        vw.write(f)
    vw.release()


In [3]:
def find_wave_fronts(gray: np.ndarray, k: int = 10, min_dist: int = 10):
    """
    Pick up to `k` high-contrast points spread at least `min_dist` pixels apart.
    """
    # gradient magnitude as "image" for corner detection
    gx = cv.Sobel(gray, cv.CV_32F, 1, 0, ksize=3)
    gy = cv.Sobel(gray, cv.CV_32F, 0, 1, ksize=3)
    gmag = cv.magnitude(gx, gy)                    # float32

    # goodFeaturesToTrack requires 8-bit or float; gmag already float
    corners = cv.goodFeaturesToTrack(gmag,
                                     maxCorners=k,
                                     qualityLevel=0.01,
                                     minDistance=min_dist,
                                     blockSize=3,
                                     useHarrisDetector=False)
    return corners.astype(np.float32)   # shape (≤k,1,2)


In [4]:
def track_wave(stack: np.ndarray,
               step: int      = 5,
               win_size: int  = 100,
               seeds: int     = 10,
               out_prefix: str = "wave"):
    """
    Track wave-front points and return hop velocities (px/frame).

    * No early-stop logic: runs through the entire stack.
    * “Dead” points are dropped forever once LK fails to find them.
    """
    STEP = max(1, step)
    u8   = to_uint8(stack)                                   # (T,H,W) → uint8
    bgr  = [cv.cvtColor(fr, cv.COLOR_GRAY2BGR) for fr in u8]

    # ─── Lucas–Kanade parameters ───
    lk = dict(winSize=(win_size, win_size),
              maxLevel=1,
              criteria=(cv.TERM_CRITERIA_EPS |
                        cv.TERM_CRITERIA_COUNT, 10, 0.03))

    # ─── initial seed points ───
    old_gray = u8[0]
    p0 = find_wave_fronts(old_gray, k=seeds)                 # (N,1,2) float32
    active_keys = list(range(len(p0)))                       # 0…N-1

    # ─── drawing setup ───
    mask   = np.zeros_like(bgr[0])
    colour = np.random.randint(0, 255, (len(p0), 3))
    frames = []

    # track history per point: idx → list[(frame#, (x,y))]
    hist = {idx: [(0, tuple(pt[0]))] for idx, pt in enumerate(p0)}

    # ─── main loop over frames ───
    for f in range(1, len(u8)):
        new_gray = u8[f]
        p1, st, _ = cv.calcOpticalFlowPyrLK(old_gray, new_gray, p0, None, **lk)
        if p1 is None:
            break

        st = st.reshape(-1)
        good_idx = np.where(st == 1)[0]                      # survivors
        good_new = p1[st == 1].reshape(-1, 2)                # (M,2)

        frame = bgr[f].copy()
        for local_i, (x, y) in enumerate(good_new):
            idx = active_keys[good_idx[local_i]]             # global index
            x_prev, y_prev = hist[idx][-1][1]
            hist[idx].append((f, (x, y)))

            mask = cv.line(mask, (int(x_prev), int(y_prev)),
                                 (int(x),     int(y)),
                                 colour[idx].tolist(), 2)
            frame = cv.circle(frame, (int(x), int(y)),
                              4, colour[idx].tolist(), -1)

        frames.append(cv.add(frame, mask))
        old_gray = new_gray.copy()

        # ----- permanently remove lost points -----
        p0 = good_new.reshape(-1, 1, 2)
        active_keys = [active_keys[i] for i in good_idx]

        if len(p0) == 0:         # everything died → break early
            break

    # ─── compute velocities every STEP frames ───
    velocities = []
    for h in hist.values():
        for j in range(STEP, len(h)):
            fn_now,(x_now,y_now)   = h[j]
            fn_prev,(x_prev,y_prev)= h[j-STEP]
            if fn_now - fn_prev == STEP:
                velocities.append(np.hypot(x_now - x_prev,
                                           y_now - y_prev) / STEP)
    velocities = np.asarray(velocities)

    # ─── save overlay video ───
    vid_path = f"{out_prefix}_tracks.mp4"
    save_mp4([bgr[0]] + frames, vid_path)
    print(f"Overlay video saved  →  {vid_path}")

    return velocities


In [6]:
import numpy as np
import pandas as pd
from pde_utils import correlated_gaussian_field, laplacian

# ╭──────────────────────── simulation settings ───────────────────────╮
k0, k1, k2 = 0.00625, 0.3125, 1
k3, k4, k5 = 0.0625, 0.05625, 0.0625
k6, k7, k8 = 0.02083, 0.001875, 0.14062
k9, k10    = 0.25, 0.025
Drt, Drd   = 0.08, 0.4
sigma, s   = 0.75, 4
f          = 10               # dW update frequency (s)
alpha = beta = 1

size      = 100               # grid size
dt        = 0.01               # time step (s)
t_total   = 1000.0            # total run time (s)
frame_int = 0.1                # saved-frame spacing (s)

save_every = int(frame_int / dt)
dW_update  = int(f / dt)
n_steps    = int(t_total / dt)

def reaction(A, B, C):
    return (k0 + alpha*k1*A**3/(1 + k2*A**2))*B - (k3 + k4*(1+beta)*C)*A

def run_simulation(Df: float, seed: int = 42) -> np.ndarray:
    """Return stack (n_frames, H, W) for a given fuel diffusion `Df`."""
    np.random.seed(seed)
    RT = 0.1 + 0.9 * np.random.rand(size, size)
    RD = np.full((size, size), 0.1)
    F  = np.zeros((size, size))
    dW = correlated_gaussian_field(sigma, s, (size, size), 1.0)

    frames = []
    for i in range(n_steps):
        R = reaction(RT, RD, F)
        RT += dt * (R + Drt * laplacian(RT))
        RD += dt * (k5 - k6*RD - R + Drd * laplacian(RD))
        F  += dt * (k7 + k8*RT**2/(1 + k9*RT**2) - k10*dW*F + Df * laplacian(F))

        if i % dW_update == 0:
            dW = correlated_gaussian_field(sigma, s, (size, size), 1.0)
        if i % save_every == 0:
            frames.append(RT.copy())
    return np.stack(frames, axis=0)          # shape (Tsave, H, W)
# ╰─────────────────────────────────────────────────────────────────────╯

# diffusion constants to test
Df_list = [0.001,0.01,0.5]

records = []

for Df in Df_list:
    print(f"\n>>> Running Df = {Df}")
    stack = run_simulation(Df)

    # optional burn-in: skip the first 5 000 saved frames
    midpoint = stack.shape[0] // 2
    stack    = stack[midpoint:]                     

    vel = track_wave(stack,
                     step=3,            # distance every 3 saved frames
                     win_size=5,
                     seeds=6,
                     out_prefix=f"Df{Df:.2f}")

    # ----------- summary statistics -----------
    mean_v   = np.mean(vel)
    median_v = np.median(vel)
    q25_v    = np.percentile(vel, 25)
    q75_v    = np.percentile(vel, 75)
    std_v    = np.std(vel)
    min_v    = vel.min()
    max_v    = vel.max()

    print(f"Mean   = {mean_v:.3f}  px/frame")
    print(f"Median = {median_v:.3f}  px/frame")
    print(f"Q25–Q75= {q25_v:.3f} – {q75_v:.3f}  px/frame")
    print(f"Std    = {std_v:.3f}  px/frame")
    print(f"Min, Max = {min_v:.3f}, {max_v:.3f} px/frame")

    records.append({
        "Df"   : Df,
        "mean" : mean_v,
        "median": median_v,
        "q25"  : q25_v,
        "q75"  : q75_v,
        "std"  : std_v,
        "min"  : min_v,
        "max"  : max_v

    })
    
# results as a Pandas DataFrame
results_df = pd.DataFrame(records)
results_df





>>> Running Df = 0.001
Overlay video saved  →  Df0.00_tracks.mp4
Mean   = 0.016  px/frame
Median = 0.014  px/frame
Q25–Q75= 0.010 – 0.019  px/frame
Std    = 0.010  px/frame
Min, Max = 0.000, 0.170 px/frame

>>> Running Df = 0.01
Overlay video saved  →  Df0.01_tracks.mp4
Mean   = 0.018  px/frame
Median = 0.015  px/frame
Q25–Q75= 0.011 – 0.019  px/frame
Std    = 0.072  px/frame
Min, Max = 0.000, 4.362 px/frame

>>> Running Df = 0.5
Overlay video saved  →  Df0.50_tracks.mp4
Mean   = 0.004  px/frame
Median = 0.003  px/frame
Q25–Q75= 0.002 – 0.004  px/frame
Std    = 0.004  px/frame
Min, Max = 0.000, 0.073 px/frame


,Df,mean,median,q25,q75,std,min,max
0,0.001,0.016010,0.014249,0.010219,0.019105,0.010231,0.000157,0.169684
1,0.010,0.017687,0.014815,0.010857,0.018978,0.071907,0.000270,4.362416
2,0.500,0.003680,0.002689,0.001544,0.004448,0.004056,0.000010,0.072947
